# CATSA 3-Class (Rest / Distress / Deep Flow)

이 노트북은 YAML 없이 한 파일에서 바로 학습합니다.

## Label Mapping (Nback 제외)
- `0`: Rest -> `Baseline`
- `1`: Distress -> `Stroop`
- `2`: DeepFlow -> `Logic`, `Sudoku`

## Multi-Branch Late Fusion (Raw BVP branch 제거)
- ACC(32Hz) 입력: `(B, 3, 1920)`
- Slow group(4Hz) 입력: `(B, 4, 240)` where channels = `[EDA, TEMP, HR, HRV]`
- HR/HRV는 `BVP(64Hz)`에서 추출하되, 원시 BVP는 모델 입력에서 제외

브랜치 출력:
- ACC branch: `(B, 32, 240)` via stride `4 -> 2`
- Slow branch: `(B, 16, 240)` via stride `1`

Fusion:
- concat -> `(B, 48, 240)`
- permute -> `(B, 240, 48)`
- Transformer Encoder -> GAP -> FC(3)

## Dynamic Stride (Train only)
- Rest: `10s`
- Distress: `5s`
- DeepFlow: `20s`


In [1]:
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.signal import find_peaks
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import Dataset, DataLoader

# ===================== User Config =====================
DATASET_ROOT = Path('/home/binghin2/Myproject/Dataset/CATSA')

TASK_TO_CLASS = {
    'Baseline': 0,
    'Stroop': 1,
    'Logic': 2,
    'Sudoku': 2,
}
CLASS_NAMES = ['Rest', 'Distress', 'DeepFlow']

WINDOW_SEC = 60
STRIDE_SEC_EVAL = 60

# Dynamic stride for train split (class-wise balancing)
TRAIN_CLASS_STRIDE_SEC = {
    0: 10,  # Rest
    1: 5,   # Distress (oversample)
    2: 20,  # DeepFlow (undersample)
}

# Native sampling rates
FS_BVP = 64
FS_ACC = 32
FS_SLOW = 4   # EDA/TEMP and derived HR/HRV

# 60 sec expected lengths
LEN_ACC = WINDOW_SEC * FS_ACC   # 1920
LEN_SLOW = WINDOW_SEC * FS_SLOW # 240

BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42

# Overfitting control
DROPOUT = 0.4
EARLY_STOP_PATIENCE = 8
EARLY_STOP_F1_MIN_DELTA = 1e-4

# Focal loss config
FOCAL_GAMMA = 2.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print('Device:', DEVICE)
print('Dataset root:', DATASET_ROOT)
print('Dropout:', DROPOUT)
print('Focal loss:', {'gamma': FOCAL_GAMMA, 'alpha': 'class_weights'})
print('Early stopping monitor: val_macro_f1', {'patience': EARLY_STOP_PATIENCE, 'min_delta': EARLY_STOP_F1_MIN_DELTA})
print('Train class strides (sec):', TRAIN_CLASS_STRIDE_SEC)


Device: cuda
Dataset root: /home/binghin2/Myproject/Dataset/CATSA
Dropout: 0.4
Focal loss: {'gamma': 2.0, 'alpha': 'class_weights'}
Early stopping monitor: val_macro_f1 {'patience': 8, 'min_delta': 0.0001}
Train class strides (sec): {0: 10, 1: 5, 2: 20}


In [2]:
def read_csv_array(path: Path) -> np.ndarray:
    return pd.read_csv(path).values.astype(np.float32)


def bvp_to_hr_hrv_4hz(bvp_64: np.ndarray, fs: int = 64, out_fs: int = 4,
                     min_peak_distance_sec: float = 0.30,
                     hrv_window_sec: int = 10):
    """
    BVP(64Hz)에서 피크를 찾고 HR, HRV(IBI rolling std)를 계산한 뒤 4Hz로 보간.
    Returns: hr_4hz (T4,), hrv_4hz (T4,)
    """
    sig = np.asarray(bvp_64, dtype=np.float32).reshape(-1)
    min_dist = max(1, int(fs * min_peak_distance_sec))
    peaks, _ = find_peaks(sig, distance=min_dist)

    n4 = len(sig) // (fs // out_fs)
    t4 = np.arange(n4) / float(out_fs)

    if len(peaks) < 3:
        return np.zeros(n4, dtype=np.float32), np.zeros(n4, dtype=np.float32)

    t_peaks = peaks / float(fs)
    ibi = np.diff(t_peaks)
    ibi = np.clip(ibi, 1e-3, None)
    hr = 60.0 / ibi

    t_ibi = t_peaks[1:]
    hr_4 = np.interp(t4, t_ibi, hr).astype(np.float32)
    ibi_series = pd.Series(ibi)
    hrv = ibi_series.rolling(window=max(2, int(hrv_window_sec)), min_periods=1).std().fillna(0).values
    hrv_4 = np.interp(t4, t_ibi, hrv).astype(np.float32)

    return hr_4, hrv_4


def load_task_modalities(subject_dir: Path, task: str):
    task_dir = subject_dir / task
    paths = {
        'acc': task_dir / 'ACC.csv',
        'bvp': task_dir / 'BVP.csv',
        'eda': task_dir / 'EDA.csv',
        'temp': task_dir / 'TEMP.csv',
    }
    if not all(p.exists() for p in paths.values()):
        return None

    acc = read_csv_array(paths['acc'])
    bvp = read_csv_array(paths['bvp']).reshape(-1)
    eda = read_csv_array(paths['eda']).reshape(-1)
    temp = read_csv_array(paths['temp']).reshape(-1)

    # ACC 3축 보장
    if acc.ndim == 1:
        acc = acc.reshape(-1, 1)
    if acc.shape[1] < 3:
        acc = np.tile(acc, (1, 3))[:, :3]
    else:
        acc = acc[:, :3]

    # BVP -> HR/HRV (4Hz). Raw BVP itself is not used as model input.
    hr_4, hrv_4 = bvp_to_hr_hrv_4hz(bvp, fs=FS_BVP, out_fs=FS_SLOW)

    # 시간축 정렬: 4Hz 기준 최소 길이
    t4_from_acc = len(acc) // (FS_ACC // FS_SLOW)
    t4_from_bvp = len(bvp) // (FS_BVP // FS_SLOW)
    t4 = min(len(eda), len(temp), len(hr_4), len(hrv_4), t4_from_acc, t4_from_bvp)
    if t4 < LEN_SLOW:
        return None

    eda = eda[:t4]
    temp = temp[:t4]
    hr_4 = hr_4[:t4]
    hrv_4 = hrv_4[:t4]
    acc = acc[: t4 * (FS_ACC // FS_SLOW)]

    slow = np.stack([eda, temp, hr_4, hrv_4], axis=1).astype(np.float32)  # (T4,4)

    return {
        'acc': acc.astype(np.float32),   # (T32,3)
        'slow': slow,                     # (T4,4)
    }


def create_windows(mods: dict, label: int, stride_sec: int):
    """
    Returns list of dict with channel-first tensors ready for Conv1d:
    - acc:  (3, 1920)
    - slow: (4, 240)
    """
    stride4 = stride_sec * FS_SLOW
    w4 = LEN_SLOW
    w32 = LEN_ACC
    ratio32 = FS_ACC // FS_SLOW  # 8

    n4 = len(mods['slow'])
    out = []
    start4 = 0
    while start4 + w4 <= n4:
        s32 = start4 * ratio32

        slow_win = mods['slow'][start4:start4 + w4]           # (240,4)
        acc_win = mods['acc'][s32:s32 + w32]                  # (1920,3)

        if len(slow_win) != w4 or len(acc_win) != w32:
            break

        out.append({
            'slow': slow_win.T.copy(),  # (4,240)
            'acc': acc_win.T.copy(),    # (3,1920)
            'y': int(label),
        })
        start4 += stride4

    return out


def list_subjects(root: Path):
    return sorted([p.name for p in root.glob('Sub*') if p.is_dir()])


def split_subjects(subjects, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(subjects))
    n = len(subjects)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]

    train_subjects = [subjects[i] for i in train_idx]
    val_subjects = [subjects[i] for i in val_idx]
    test_subjects = [subjects[i] for i in test_idx]
    return train_subjects, val_subjects, test_subjects


def build_split_windows(subjects, is_train=True):
    all_windows = []

    for s in subjects:
        sdir = DATASET_ROOT / s
        for task, cls in TASK_TO_CLASS.items():
            mods = load_task_modalities(sdir, task)
            if mods is None:
                continue
            stride = TRAIN_CLASS_STRIDE_SEC[cls] if is_train else STRIDE_SEC_EVAL
            all_windows.extend(create_windows(mods, cls, stride_sec=stride))

    return all_windows


subjects = list_subjects(DATASET_ROOT)
tr_subj, va_subj, te_subj = split_subjects(subjects, seed=SEED)

print(f'Subject split -> train:{len(tr_subj)} val:{len(va_subj)} test:{len(te_subj)}')


Subject split -> train:35 val:7 test:8


In [3]:
class CATSAMultiModalDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        return (
            torch.tensor(w['acc'], dtype=torch.float32),   # (3,1920)
            torch.tensor(w['slow'], dtype=torch.float32),  # (4,240)
            torch.tensor(w['y'], dtype=torch.long),
        )


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class MultiBranchCNNTransformer(nn.Module):
    """
    Input:
      acc  = (B, 3, 1920)
      slow = (B, 4, 240)  [EDA, TEMP, HR, HRV]

    Branch output:
      acc_f  = (B, 32, 240)  stride 4 -> 2
      slow_f = (B, 16, 240)  stride 1

    Fusion:
      cat -> (B, 48, 240)
      permute -> (B, 240, 48)
    """
    def __init__(self, n_classes=3, d_model=48, nhead=8, nlayer=2, ff_dim=192, dropout=0.4):
        super().__init__()

        self.acc_branch = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=9, stride=4, padding=4),   # 1920 -> 480
            nn.BatchNorm1d(16),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(16, 32, kernel_size=9, stride=2, padding=4),  # 480 -> 240
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.slow_branch = nn.Sequential(
            nn.Conv1d(4, 16, kernel_size=5, stride=1, padding=2),   # 240 -> 240
            nn.BatchNorm1d(16),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(16, 16, kernel_size=3, stride=1, padding=1),  # 240 -> 240
            nn.BatchNorm1d(16),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.posenc = PositionalEncoding(d_model=d_model, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=nlayer)
        self.norm = nn.LayerNorm(d_model)
        self.cls = nn.Linear(d_model, n_classes)

    def forward(self, acc, slow, return_features=False):
        acc_f = self.acc_branch(acc)    # (B,32,240)
        slow_f = self.slow_branch(slow) # (B,16,240)

        fused = torch.cat([acc_f, slow_f], dim=1)       # (B,48,240)
        seq = fused.permute(0, 2, 1).contiguous()       # (B,240,48)

        x = self.posenc(seq)
        x = self.transformer(x)
        x = self.norm(x)
        pooled = x.mean(dim=1)
        logits = self.cls(pooled)

        if return_features:
            return logits, seq
        return logits


# Shape sanity check
with torch.no_grad():
    m = MultiBranchCNNTransformer(dropout=DROPOUT).to(DEVICE)
    acc = torch.randn(2, 3, LEN_ACC, device=DEVICE)
    slow = torch.randn(2, 4, LEN_SLOW, device=DEVICE)
    logits, seq = m(acc, slow, return_features=True)
    print('Expected seq shape: (2, 240, 48)')
    print('Actual seq shape  :', tuple(seq.shape))
    print('Logits shape      :', tuple(logits.shape))


/home/binghin2/miniconda3/envs/myenv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Expected seq shape: (2, 240, 48)
Actual seq shape  : (2, 240, 48)
Logits shape      : (2, 3)


In [4]:
def fit_channel_stats(train_windows, key):
    """Compute per-channel mean/std from training windows (channel-first)."""
    x = np.concatenate([w[key] for w in train_windows], axis=1)  # (C, total_T)
    mean = x.mean(axis=1, keepdims=True)
    std = x.std(axis=1, keepdims=True) + 1e-6
    return mean.astype(np.float32), std.astype(np.float32)


def apply_channel_stats(windows, stats):
    out = []
    for w in windows:
        nw = dict(w)
        for key, (mean, std) in stats.items():
            nw[key] = ((w[key] - mean) / std).astype(np.float32)
        out.append(nw)
    return out


def class_weights_from_windows(train_windows, n_classes=3):
    y = np.array([w['y'] for w in train_windows], dtype=np.int64)
    cnt = np.bincount(y, minlength=n_classes).astype(np.float32)
    total = cnt.sum()
    w = total / (n_classes * np.maximum(cnt, 1.0))
    return torch.tensor(w, dtype=torch.float32)


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(logits, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss


def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_pred, all_true = [], []

    with torch.no_grad():
        for acc, slow, y in loader:
            acc = acc.to(device)
            slow = slow.to(device)
            y = y.to(device)

            logits = model(acc, slow)
            loss = criterion(logits, y)

            total_loss += loss.item() * y.size(0)
            pred = logits.argmax(dim=1)
            all_pred.extend(pred.cpu().numpy().tolist())
            all_true.extend(y.cpu().numpy().tolist())

    n = max(len(all_true), 1)
    acc = float(np.mean(np.array(all_pred) == np.array(all_true))) if len(all_true) > 0 else 0.0
    macro_f1 = f1_score(all_true, all_pred, average='macro', zero_division=0) if len(all_true) > 0 else 0.0
    return total_loss / n, acc, macro_f1, all_pred, all_true


def print_class_counts(windows, title):
    y = np.array([w['y'] for w in windows], dtype=np.int64)
    cnt = np.bincount(y, minlength=len(CLASS_NAMES))
    msg = ', '.join([f"{CLASS_NAMES[i]}:{int(cnt[i])}" for i in range(len(CLASS_NAMES))])
    print(f"{title}: {msg}")


def run_training():
    print('Building windows...')
    tr_win = build_split_windows(tr_subj, is_train=True)
    va_win = build_split_windows(va_subj, is_train=False)
    te_win = build_split_windows(te_subj, is_train=False)

    print(f'Windows -> train:{len(tr_win)} val:{len(va_win)} test:{len(te_win)}')
    print_class_counts(tr_win, 'Train class windows (dynamic stride)')
    print_class_counts(va_win, 'Val class windows')
    print_class_counts(te_win, 'Test class windows')

    if len(tr_win) == 0 or len(va_win) == 0 or len(te_win) == 0:
        raise RuntimeError('One of dataset splits has zero windows. Check dataset path and files.')

    # Normalize using train split only
    stats = {
        'acc': fit_channel_stats(tr_win, 'acc'),
        'slow': fit_channel_stats(tr_win, 'slow'),
    }
    tr_win = apply_channel_stats(tr_win, stats)
    va_win = apply_channel_stats(va_win, stats)
    te_win = apply_channel_stats(te_win, stats)

    train_loader = DataLoader(CATSAMultiModalDataset(tr_win), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(CATSAMultiModalDataset(va_win), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(CATSAMultiModalDataset(te_win), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = MultiBranchCNNTransformer(n_classes=3, dropout=DROPOUT).to(DEVICE)
    class_weights = class_weights_from_windows(tr_win, n_classes=3).to(DEVICE)
    criterion = FocalLoss(alpha=class_weights, gamma=FOCAL_GAMMA)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    # Early stopping tracks validation macro-F1
    best_state = None
    best_val_macro_f1 = 0.0
    best_val_acc = 0.0
    best_val_loss = float('inf')
    patience_counter = 0

    print('\nEpoch | TrainLoss TrainAcc | ValLoss ValAcc ValMacroF1 | ES')
    print('-' * 72)

    for ep in range(1, EPOCHS + 1):
        model.train()
        tr_loss_sum = 0.0
        tr_true, tr_pred = [], []

        for acc, slow, y in train_loader:
            acc = acc.to(DEVICE)
            slow = slow.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(acc, slow)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            tr_loss_sum += loss.item() * y.size(0)
            tr_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())
            tr_true.extend(y.detach().cpu().numpy().tolist())

        scheduler.step()

        tr_n = max(len(tr_true), 1)
        tr_loss = tr_loss_sum / tr_n
        tr_acc = float(np.mean(np.array(tr_pred) == np.array(tr_true)))

        va_loss, va_acc, va_macro_f1, _, _ = eval_model(model, val_loader, criterion, DEVICE)

        improved = va_macro_f1 > (best_val_macro_f1 + EARLY_STOP_F1_MIN_DELTA)
        if improved:
            best_val_macro_f1 = va_macro_f1
            best_val_acc = va_acc
            best_val_loss = va_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            mark = ' *'
        else:
            patience_counter += 1
            mark = ''

        print(f'{ep:5d} | {tr_loss:8.4f} {tr_acc:8.2%} | {va_loss:7.4f} {va_acc:7.2%} {va_macro_f1:10.3f} | {patience_counter:2d}/{EARLY_STOP_PATIENCE}{mark}')

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'Early stopping triggered at epoch {ep} (best val macro-F1: {best_val_macro_f1:.3f})')
            break

    print('\nBest val macro-F1:', f'{best_val_macro_f1:.3f}')
    print('Best val acc     :', f'{best_val_acc:.2%}')
    print('Best val loss    :', f'{best_val_loss:.4f}')
    if best_state is not None:
        model.load_state_dict(best_state)

    te_loss, te_acc, te_macro_f1, te_pred, te_true = eval_model(model, test_loader, criterion, DEVICE)
    print('Test loss         :', f'{te_loss:.4f}')
    print('Test acc          :', f'{te_acc:.2%}')
    print('Test macro-F1     :', f'{te_macro_f1:.3f}')

    print('\nClassification Report (Test)')
    print(classification_report(te_true, te_pred, target_names=CLASS_NAMES, zero_division=0))

    cm = confusion_matrix(te_true, te_pred, labels=[0, 1, 2])
    cm_df = pd.DataFrame(cm, index=[f'True_{c}' for c in CLASS_NAMES], columns=[f'Pred_{c}' for c in CLASS_NAMES])
    print('Confusion Matrix (Test)')
    display(cm_df)

    return model


model = run_training()


Building windows...
Windows -> train:1820 val:84 test:96
Train class windows (dynamic stride): Rest:455, Distress:875, DeepFlow:490
Val class windows: Rest:21, Distress:21, DeepFlow:42
Test class windows: Rest:24, Distress:24, DeepFlow:48


/home/binghin2/miniconda3/envs/myenv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



Epoch | TrainLoss TrainAcc | ValLoss ValAcc ValMacroF1 | ES
------------------------------------------------------------------------
    1 |   0.4918   35.49% |  0.7824  32.14%      0.315 |  0/8 *
    2 |   0.4558   41.70% |  0.6629  32.14%      0.271 |  1/8
    3 |   0.4446   44.29% |  0.6164  38.10%      0.346 |  0/8 *
    4 |   0.4451   43.19% |  0.6656  41.67%      0.374 |  0/8 *
    5 |   0.4332   44.12% |  0.6221  39.29%      0.366 |  1/8
    6 |   0.4391   44.07% |  1.0815  30.95%      0.251 |  2/8
    7 |   0.4305   44.40% |  0.6069  29.76%      0.229 |  3/8
    8 |   0.4310   42.75% |  0.6633  25.00%      0.133 |  4/8
    9 |   0.4220   45.99% |  0.6505  39.29%      0.394 |  0/8 *
   10 |   0.4193   44.51% |  0.8214  27.38%      0.230 |  1/8
   11 |   0.4145   45.99% |  0.7215  28.57%      0.257 |  2/8
   12 |   0.4167   46.04% |  0.6697  30.95%      0.287 |  3/8
   13 |   0.4180   45.60% |  0.6564  34.52%      0.328 |  4/8
   14 |   0.4075   47.31% |  0.6310  35.71%      0.3

,Pred_Rest,Pred_Distress,Pred_DeepFlow
True_Rest,19,4,1
True_Distress,13,10,1
True_DeepFlow,31,16,1


In [6]:
# Save trained model checkpoint
from pathlib import Path
from datetime import datetime
import json

save_dir = Path('/home/binghin2/Myproject/Research/CATSA/Train/CNN-TF/Save_model')
save_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
ckpt_path = save_dir / f'cnn_tf_3class_nobvp_{ts}.pth'
meta_path = save_dir / f'cnn_tf_3class_nobvp_{ts}_meta.json'

checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_names': CLASS_NAMES,
    'task_to_class': TASK_TO_CLASS,
    'window_sec': WINDOW_SEC,
    'train_class_stride_sec': TRAIN_CLASS_STRIDE_SEC,
    'stride_sec_eval': STRIDE_SEC_EVAL,
    'sampling_rates': {
        'acc': FS_ACC,
        'slow': FS_SLOW,
        'bvp_source_for_hr_hrv': FS_BVP,
    },
    'lengths': {
        'acc': LEN_ACC,
        'slow': LEN_SLOW,
    },
    'optimizer': {
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'seed': SEED,
    },
    'dropout': DROPOUT,
    'loss': {
        'name': 'FocalLoss',
        'gamma': FOCAL_GAMMA,
        'alpha': 'class_weights',
    },
    'early_stopping': {
        'monitor': 'val_macro_f1',
        'patience': EARLY_STOP_PATIENCE,
        'min_delta': EARLY_STOP_F1_MIN_DELTA,
    },
    'raw_bvp_branch_used': False,
}

torch.save(checkpoint, ckpt_path)

with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump({
        'checkpoint_path': str(ckpt_path),
        'created_at': ts,
        'device': str(DEVICE),
        'num_classes': len(CLASS_NAMES),
        'classes': CLASS_NAMES,
        'loss': 'FocalLoss',
        'early_stopping_monitor': 'val_macro_f1',
        'raw_bvp_branch_used': False,
    }, f, ensure_ascii=False, indent=2)

print('Saved checkpoint:', ckpt_path)
print('Saved metadata  :', meta_path)


Saved checkpoint: /home/binghin2/Myproject/Research/CATSA/Train/CNN-TF/Save_model/cnn_tf_3class_nobvp_20260313_134841.pth
Saved metadata  : /home/binghin2/Myproject/Research/CATSA/Train/CNN-TF/Save_model/cnn_tf_3class_nobvp_20260313_134841_meta.json


In [ ]:
# Evaluation summary on test split: Accuracy, Macro F1, Confusion Matrix + Bar Plot
import matplotlib.pyplot as plt

# 1) Rebuild train/test windows to apply identical train-based normalization
tr_eval = build_split_windows(tr_subj, is_train=True)
te_eval = build_split_windows(te_subj, is_train=False)

stats_eval = {
    'acc': fit_channel_stats(tr_eval, 'acc'),
    'slow': fit_channel_stats(tr_eval, 'slow'),
}
te_eval = apply_channel_stats(te_eval, stats_eval)

# 2) Inference on test split
te_loader = DataLoader(CATSAMultiModalDataset(te_eval), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for acc, slow, y in te_loader:
        acc = acc.to(DEVICE)
        slow = slow.to(DEVICE)
        logits = model(acc, slow)
        pred = logits.argmax(dim=1).cpu().numpy().tolist()

        y_pred.extend(pred)
        y_true.extend(y.numpy().tolist())

# 3) Metrics
acc_score = float(np.mean(np.array(y_pred) == np.array(y_true))) if len(y_true) > 0 else 0.0
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0) if len(y_true) > 0 else 0.0

print(f"Test Accuracy : {acc_score:.4f} ({acc_score*100:.2f}%)")
print(f"Test Macro F1 : {macro_f1:.4f} ({macro_f1*100:.2f}%)")

# 4) Bar plot for overall metrics
metric_names = ['Accuracy', 'Macro F1']
metric_vals = [acc_score * 100, macro_f1 * 100]
colors = ['#4C78A8', '#F58518']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(metric_names, metric_vals, color=colors, width=0.6)
ax.set_ylim(0, 100)
ax.set_ylabel('Score (%)')
ax.set_title('Test Metrics (CNN-TF)')
ax.grid(axis='y', alpha=0.25)

for bar, val in zip(bars, metric_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 1.0, f'{val:.2f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# 5) Confusion Matrix (table + heatmap)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
cm_df = pd.DataFrame(cm, index=[f"True_{c}" for c in CLASS_NAMES], columns=[f"Pred_{c}" for c in CLASS_NAMES])

print("\nConfusion Matrix")
display(cm_df)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(np.arange(len(CLASS_NAMES)))
ax.set_yticks(np.arange(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix Heatmap')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black')

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print("\nClassification Report")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

Test Accuracy : 0.3125 (31.25%)
Test Macro F1 : 0.2821

Confusion Matrix


,Pred_Rest,Pred_Distress,Pred_DeepFlow
True_Rest,19,4,1
True_Distress,13,10,1
True_DeepFlow,31,16,1



Classification Report
              precision    recall  f1-score   support

        Rest       0.30      0.79      0.44        24
    Distress       0.33      0.42      0.37        24
    DeepFlow       0.33      0.02      0.04        48

    accuracy                           0.31        96
   macro avg       0.32      0.41      0.28        96
weighted avg       0.33      0.31      0.22        96

